In [0]:
%sql
-- 9: Merchants ranked by fraud rate (CTE + LEFT JOIN + window ranking)
WITH merchant_txns AS (
  SELECT merchant_id, COUNT(*) AS txn_count
  FROM fintech_fraud_risk.gold.fact_transactions
  GROUP BY merchant_id
),
merchant_fraud AS (
  SELECT f.merchant_id, COUNT(DISTINCT fa.alert_id) AS fraud_count
  FROM fintech_fraud_risk.gold.fact_transactions f
  JOIN fintech_fraud_risk.silver.fraud_alerts fa ON f.transaction_id = fa.transaction_id
  GROUP BY f.merchant_id
)
SELECT
  m.merchant_id, dm.merchant_name, t.txn_count, COALESCE(mf.fraud_count, 0) AS fraud_count,
  ROUND(COALESCE(mf.fraud_count, 0) * 100.0 / t.txn_count, 2) AS fraud_rate_pct,
  RANK() OVER (ORDER BY COALESCE(mf.fraud_count, 0) * 1.0 / t.txn_count DESC) AS risk_rank
FROM merchant_txns t
JOIN fintech_fraud_risk.gold.dim_merchant dm ON t.merchant_id = dm.merchant_id
LEFT JOIN merchant_fraud mf ON t.merchant_id = mf.merchant_id
LEFT JOIN merchant_txns m ON m.merchant_id = t.merchant_id
WHERE t.txn_count > 20   -- exclude low-volume merchants from skewing the rate
ORDER BY risk_rank
LIMIT 20;


In [0]:
%sql

-- 10: Chargeback rate by merchant category
SELECT
  dm.merchant_category,
  COUNT(DISTINCT f.transaction_id) AS txn_count,
  COUNT(DISTINCT cb.chargeback_id) AS chargeback_count,
  ROUND(COUNT(DISTINCT cb.chargeback_id) * 100.0 / COUNT(DISTINCT f.transaction_id), 3) AS chargeback_rate_pct
FROM fintech_fraud_risk.gold.fact_transactions f
JOIN fintech_fraud_risk.gold.dim_merchant dm ON f.merchant_id = dm.merchant_id
LEFT JOIN fintech_fraud_risk.silver.chargebacks cb ON f.transaction_id = cb.transaction_id
GROUP BY dm.merchant_category
ORDER BY chargeback_rate_pct DESC;